Visualize precomputed activation distributions, split by a given token, with an extension to filter by a specific SMARTS pattern.

In [ ]:
# pkgs
import h5py
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tqdm.notebook import tqdm
from matplotlib.patches import Patch

from intermol.main.utils import load_model_from_HF
from intermol.interp.molecular_concepts import map_atom_idx_to_token_idx
from intermol.interp.utils import h5_chunk_sorter

# defaults
LAYERS = [1, 3, 6, 9, 12]
DIRS = {layer: next(Path("..").glob(f"results_layer{layer}_*")) for layer in LAYERS}

DATADIR = Path("../data/mols")
SAMPLE_PTH = DATADIR / "valid_25e4.txt" # contains canonical SMILES
SAMPLE_ACT_NAME = "valid_acts.h5"

OUTDIR = Path("../plots")


# init tokenizer
tokenizer = load_model_from_HF("ibm/MoLFormer-XL-both-10pct", tokenizer_only=True)

# init samples
with open(SAMPLE_PTH, "r") as h:
    samples = [smi.rstrip('\n') for smi in h.readlines()]

In [ ]:
# cfgs
LAYER = 12
FEATURE = 3004
SMARTS = None # optional

# main colls
activations = []
token_ids = []

# build SMARTS
if SMARTS is not None:
    from rdkit import Chem

    ptrn = Chem.MolFromSmarts(SMARTS)
    tks_ptrn = []


# main
with h5py.File(DIRS[LAYER] / SAMPLE_ACT_NAME, "r") as h5f:
    chunks = h5_chunk_sorter(list(h5f.keys()))
    n_features = h5f.attrs["num_features"]

    curr_smi = 0
    for c in tqdm(chunks, desc="Collating activations..."):
        g = h5f[c]

        molptr = g['molptr'][:]
        indptr = g['indptr'][:]
        indices = g['indices'][:]
        data = g['data'][:]

        # colls
        tk_cnt = molptr.sum(dtype=np.int64)
        tk_acts = np.empty(tk_cnt, dtype=np.float32)
        tk_ids = np.empty(tk_cnt, dtype=np.uint16)
        if SMARTS is not None:
            is_ptrn_exists = np.zeros(tk_cnt, dtype=np.uint8)

        curr_nt = 0
        curr_indptr = 0
        curr_indices = 0
        for nt_i, nt in enumerate(molptr):
            e_nt = curr_nt + nt
            smi = samples[curr_smi]

            if SMARTS is not None:
                tokens = tokenizer.tokenize(smi)
                ai_to_ti = map_atom_idx_to_token_idx(tokens)
                ti_to_ai = {v: k for k, v in ai_to_ti.items()}

            # get token ids
            ids = np.array(
                tokenizer.encode(smi, add_special_tokens=False),
                dtype=np.uint16
            )
            tk_ids[curr_nt:e_nt] = ids

            e_indptr = curr_indptr + n_features + 1
            mol_indptr = indptr[curr_indptr:e_indptr]

            e_indices = curr_indices + mol_indptr[-1]
            mol_indices = indices[curr_indices:e_indices]
            mol_data = data[curr_indices:e_indices]

            # get activations at feature
            act = np.full(nt, -1, dtype=np.float32)
            s = mol_indptr[FEATURE]
            e = mol_indptr[FEATURE+1]

            if (e - s) > 0:
                f_data = mol_data[s:e]
                f_indices = mol_indices[s:e]
                act[f_indices] = f_data

                if SMARTS is not None:
                    mol = Chem.MolFromSmiles(smi)
                    ms = set(
                        at_idx for at_idxs in mol.GetSubstructMatches(ptrn)
                        for at_idx in at_idxs
                    )

                    tk_idxs = f_indices.tolist()
                    is_exists = [
                        tk_i for tk_i in tk_idxs
                        if (tk_i in ti_to_ai) and (ti_to_ai[tk_i] in ms)
                    ]

                    if len(is_exists) > 0:
                        exs = np.zeros(nt, dtype=np.uint8)
                        exs[is_exists] = 1
                        is_ptrn_exists[curr_nt:e_nt] = exs

            tk_acts[curr_nt:e_nt] = act

            curr_nt = e_nt
            curr_indptr = e_indptr
            curr_indices = e_indices
            curr_smi += 1

        mask = tk_acts > -1
        activations.append(tk_acts[mask])
        token_ids.append(tk_ids[mask])

        if SMARTS is not None:
            tks_ptrn.append(is_ptrn_exists[mask])

# concat
activations = pl.DataFrame((np.concatenate(activations)), schema=["norm act"])
token_ids = tokenizer.convert_ids_to_tokens(np.concatenate(token_ids))
tks_ptrn = [pl.Series("label", np.concatenate(tks_ptrn))] if SMARTS is not None else []

# to dataframe
df = activations.with_columns(pl.Series("token", token_ids), *tks_ptrn)

In [ ]:
# vis
## cfgs
TOKENS = "S"
NAME_IF_MATCH = "S in C-S-C"
NAME_IF_NOT_MATCH = "S not in C-S-C"

CMAP = sns.color_palette("Set1")[:5]

## main
if SMARTS is not None:
    sele = (
        pl.when(pl.col("token") == TOKENS)
        .then(
            pl.when(pl.col("label") == 1)
            .then(pl.lit(NAME_IF_MATCH))
            .otherwise(pl.lit(NAME_IF_NOT_MATCH))
        )
        .otherwise(pl.lit("Other tokens"))
        .alias("group")
    )
    labels = [NAME_IF_MATCH, NAME_IF_NOT_MATCH, "Other tokens"]
else:
    sele = (
        pl.when(pl.col("token") == TOKENS)
        .then(pl.lit(NAME_IF_MATCH))
        .otherwise(pl.lit(NAME_IF_NOT_MATCH))
        .alias("group")
    )
    labels = [NAME_IF_MATCH, NAME_IF_NOT_MATCH]

df_sele = df.with_columns(sele)
PALETTE = {
    label: CMAP[l_i] if label != "Other tokens" else "#cbcfd3"
    for l_i, label in labels
}

## plot
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)

sns.histplot(
    df_sele,
    x="norm act",
    hue="group",
    palette=PALETTE,
    edgecolor=None,
    legend=True,
    ax=ax
)
sns.despine()

ax.set(
    title=f"L{LAYER}/f/{FEATURE}",
    xlabel="Normalized activations"
)

ax.legend(
    handles=[
        Patch(
            color=PALETTE[l_i] if label != "Other tokens" else "#cbcfd3",
            label=label, alpha=0.6
        ) for l_i, label in enumerate(labels)
    ],
    title="",
    frameon=False,
    handlelength=0.75
)

fig.tight_layout()

## write output
fig.savefig(OUTDIR / f"actdist_L{LAYER}_f_{FEATURE}.svg", bbox_inches="tight", dpi=300)